In [0]:
dataset_moto = "/Volumes/demoworkspace/default/motostore_dataset"

In [0]:
files = dbutils.fs.ls(f"{dataset_moto}/customer-json")
display(files)

## Lecture de fichiers

In [0]:
%sql
SELECT *, _metadata.file_path
FROM read_files(
  '/Volumes/demoworkspace/default/motostore_dataset/customer-json',
  format => 'json',
  multiLine => true
)

In [0]:
%sql
DROP TABLE customer_csv;
CREATE OR REPLACE TABLE customer_csv AS
SELECT * FROM read_files(
    '/Volumes/demoworkspace/default/motostore_dataset/customer-csv/customers_4.csv',
    format => 'csv',
    header => true,
    delimiter => ',',
    schema => 'customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country STRING, updated STRING, _rescued_data STRING'
)

In [0]:
%sql
SELECT * FROM customer_csv

In [0]:
%python
(spark.read
  .table("customer_csv")
  .write
  .format("csv")
  .option('header', 'true')
  .option('delimiter', ',')
  .mode("overwrite")
  .save(f"{dataset_moto}/customer-csv/result")
)

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_moto}/customer-csv/result")
display(files)

In [0]:
%sql
SELECT * FROM customer_csv

In [0]:
%sql
REFRESH TABLE customer_csv

In [0]:
%sql
INSERT OVERWRITE customer_csv
SELECT * FROM read_files(
  '/Volumes/demoworkspace/default/motostore_dataset/customer-csv',
  format => 'csv',
  header => true,
  delimiter => ',',
  schema => 'customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country STRING, updated STRING, _rescued_data STRING'
)

In [0]:
%sql
SELECT * FROM customer_csv

## Merging Data

C00031,louis651@protonmail.com,Louis,Petrov,Female,58 Rue Victor Hugo,New York,United States,2024-08-09T00:00:00Z

In [0]:
%sql
UPDATE customer_csv
SET email = NULL
WHERE customer_id = 'C00031'

In [0]:
%sql
SELECT * FROM customer_csv

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_updates AS 
SELECT * FROM read_files(
  '/Volumes/demoworkspace/default/motostore_dataset/customer-new/customers_5.csv',
  format => 'csv',
  header => true,
  delimiter => ','
);

In [0]:
%sql
MERGE INTO customer_csv c
USING customers_updates u
ON c.customer_id = u.customer_id
WHEN MATCHED AND c.email IS NULL AND u.email IS NOT NULL THEN
  UPDATE SET email = u.email, updated = u.updated
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
CREATE OR REPLACE TABLE motocycles (
    motocycle_id STRING,
    name STRING,
    category STRING,
    engine_displacement_cc INT,
    horsepower INT,
    brand STRING,
    price NUMERIC(10,2)
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW motocycle_updates
AS SELECT * FROM read_files(
  '/Volumes/demoworkspace/default/motostore_dataset/motocycle-new/motocycles_new.json',
  format => 'json',
  multiLine => true
);

SELECT * FROM motocycle_updates

In [0]:
%sql
MERGE INTO motocycles m
USING motocycle_updates u
ON m.motocycle_id = u.motocycle_id AND m.name = u.name
WHEN NOT MATCHED AND u.category = 'Sport' THEN 
  INSERT *

## Parsing JSON Data

In [0]:
%sql
CREATE OR REPLACE TABLE customers
AS SELECT * FROM read_files(
  '/Volumes/demoworkspace/default/motostore_dataset/customer-json',
  format => 'json',
  multiLine => true
);

In [0]:
%sql
SELECT customer_id, profile.first_name, profile.adresse.country 
FROM customers

## Explode Function

In [0]:
%sql
CREATE OR REPLACE TABLE orders
AS SELECT * FROM read_files(
  '/Volumes/demoworkspace/default/motostore_dataset/order-json',
  format => 'json',
  multiLine => true
);

In [0]:
%sql
SELECT order_id, customer_id, explode(motocycles)
FROM orders

In [0]:
%sql
SELECT customer_id,
  collect_set(order_id) AS orders_set,
  collect_set(motocycles.motocycle_id) AS motcycles_set
FROM orders
GROUP BY customer_id

In [0]:
%sql
SELECT customer_id,
  collect_set(motocycles.motocycle_id) As before_flatten,
  array_distinct(flatten(collect_set(motocycles.motocycle_id))) AS after_flatten
FROM orders
GROUP BY customer_id

## Join operation

In [0]:
%sql
SELECT *, explode(motocycles) AS motocycle 
  FROM orders

In [0]:
%sql
CREATE OR REPLACE VIEW orders_enriched AS
SELECT *
FROM (
  SELECT *, explode(motocycles) AS motocycle 
  FROM orders) o
INNER JOIN motocycles m
ON o.motocycle.motocycle_id = m.motocycle_id;

SELECT * FROM orders_enriched

## Filtering Arrays

### Liste les order avec plus ou deux moto acheté

In [0]:
%sql
SELECT
  order_id,
  motocycles,
  FILTER (motocycles, i -> i.quantity >= 2) AS multiple_copies
FROM orders

# Liste les order avec plus ou deux moto acheté mais sans afficher les customers qui n'ont rien acheté.

In [0]:
%sql
SELECT order_id, multiple_copies
FROM (
  SELECT
    order_id,
    FILTER (motocycles, i -> i.quantity >= 2) AS multiple_copies
  FROM orders)
WHERE size(multiple_copies) > 0;

In [0]:
%sql
SELECT * FROM orders

## Transforming Arrays

In [0]:
%sql
SELECT
  order_id,
  motocycles,
  TRANSFORM (
    motocycles,
    b -> CAST(b.subtotal * 0.8 AS INT)
  ) AS subtotal_after_discount
FROM orders;

## User Defined Functions (UDF)

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_url(email STRING)
RETURNS STRING

RETURN concat("https://www.", split(email, "@")[1])

In [0]:
%sql
SELECT email, get_url(email) domain
FROM customer_csv

In [0]:
%sql
DESCRIBE FUNCTION EXTENDED get_url

In [0]:
%sql
CREATE FUNCTION site_type(email STRING)
RETURNS STRING
RETURN CASE 
          WHEN email like "%.com" THEN "Commercial business"
          WHEN email like "%.org" THEN "Non-profits organization"
          WHEN email like "%.edu" THEN "Educational institution"
          ELSE concat("Unknow extenstion for domain: ", split(email, "@")[1])
       END;

In [0]:
%sql
SELECT email, site_type(email) as domain_category
FROM customer_csv

In [0]:
%sql
DROP FUNCTION get_url;
DROP FUNCTION site_type;